# Fashion GAN — generating clothing images with a Conditional GAN (built from scratch)

**Goal:** train a GAN that creates **new** clothing images of a category we ask for (T-shirt, trouser, sneaker, bag, …).

**What we do**
1. Learn how a GAN works (generator vs discriminator).
2. Build the generator and **three different discriminators** from scratch with Keras layers and train them with our
   own training loop (`tf.GradientTape`) — no pretrained models, no GAN library.
3. Compare the three discriminators: **Vanilla GAN**, **Conditional GAN (cGAN)** and **AC-GAN**.
4. Train the chosen model fully and look at the **difference between input and output**.
5. Check the results with a simple classifier.

**Data:** Fashion-MNIST — 70,000 grayscale 28 × 28 images of 10 clothing categories (Zalando Research).

**Run on Google Colab:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all* (≈ 10–15 min).

## How a GAN works
A GAN has **two neural networks** that compete with each other:

| | **Generator (G)** | **Discriminator (D)** |
|---|---|---|
| Input | random noise **z** (+ the category we want) | an image (+ its category) |
| Output | a fake image | probability that the image is **real** |
| Job | fool D | catch G's fakes |

Think of G as a **forger** and D as a **detective**. Training repeats two steps:
1. **Train D** on real images (label 1 = real) and fake images from G (label 0 = fake).
2. **Train G** — generate fakes and update G so that D says "real" (label 1) for them.

Both use **binary cross-entropy** loss. As D gets better at spotting fakes, G is forced to make better fakes.
G **never sees a real image** — it learns only from D's feedback.

**Conditional GAN:** we also give the **category label** to the networks so we can choose *what* G draws.

## 1. Setup

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
import keras
from keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)
OUT_DIR = "gan_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (enable it: Runtime → Change runtime type)")

## 2. Load the data
Pixel values are scaled from `[0, 255]` to `[-1, 1]`, because the generator's last layer uses **tanh**, which outputs
values in `[-1, 1]`.

In [ ]:
CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
N_CLASSES = 10

(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train = (x_train.astype("float32") / 127.5 - 1.0)[..., None]      # (60000, 28, 28, 1) in [-1, 1]
x_test = (x_test.astype("float32") / 127.5 - 1.0)[..., None]
print("train:", x_train.shape, " test:", x_test.shape)

def to_img(x):
    # [-1, 1] → [0, 1] for plotting
    return (np.squeeze(x) + 1) / 2

fig, axes = plt.subplots(2, 10, figsize=(13, 3))
for c in range(N_CLASSES):
    for r in range(2):
        axes[r, c].imshow(to_img(x_train[np.where(y_train == c)[0][r]]), cmap="gray")
        axes[r, c].axis("off")
    axes[0, c].set_title(CLASS_NAMES[c], fontsize=8)
plt.suptitle("Real training images"); plt.tight_layout(); plt.show()

## 3. The Generator
**Input:** a noise vector **z** of 100 random numbers + a category label (0–9).
**Output:** a 28 × 28 image.

* The label goes through an **Embedding** layer (a learned vector per category) and is turned into a 7 × 7 map.
* The noise goes through a **Dense** layer and is reshaped to 7 × 7 × 128.
* Both are joined and two **Conv2DTranspose** layers (the opposite of convolution) double the size: 7 → 14 → 28.
* **BatchNormalization** keeps training stable; **LeakyReLU** is the activation; the last layer uses **tanh**.

In [ ]:
LATENT_DIM = 100

def build_generator():
    noise = keras.Input(shape=(LATENT_DIM,), name="noise")
    label = keras.Input(shape=(1,), dtype="int32", name="label")

    l = layers.Embedding(N_CLASSES, 50)(label)            # category → 50 learned numbers
    l = layers.Dense(7 * 7)(l)
    l = layers.Reshape((7, 7, 1))(l)

    n = layers.Dense(7 * 7 * 128)(noise)
    n = layers.LeakyReLU(0.2)(n)
    n = layers.Reshape((7, 7, 128))(n)

    x = layers.Concatenate()([n, l])                       # 7 × 7 × 129
    x = layers.Conv2DTranspose(128, 4, strides=2, padding="same")(x)   # 14 × 14
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Conv2DTranspose(64, 4, strides=2, padding="same")(x)    # 28 × 28
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    image = layers.Conv2D(1, 7, padding="same", activation="tanh")(x)  # 28 × 28 × 1
    return keras.Model([noise, label], image, name="generator")

build_generator().summary()

In [ ]:
# Before training, the generator only produces noise:
g = build_generator()
fake = g([tf.random.normal([10, LATENT_DIM]), tf.constant(np.arange(10)[:, None])])
fig, axes = plt.subplots(1, 10, figsize=(13, 1.6))
for i in range(10):
    axes[i].imshow(to_img(fake[i]), cmap="gray"); axes[i].axis("off"); axes[i].set_title(CLASS_NAMES[i], fontsize=7)
plt.suptitle("Untrained generator output"); plt.tight_layout(); plt.show()

## 4. Three discriminators
All three use the same simple CNN: two **Conv2D** layers with stride 2 (28 → 14 → 7), **LeakyReLU**, **Dropout**,
then **Flatten**. They differ in **how they use the category label**:

| Discriminator | Uses the label? | Output | Loss |
|---|---|---|---|
| **Vanilla GAN** (Goodfellow, 2014) | ❌ No — only sees the image | real/fake | binary cross-entropy |
| **cGAN** (Mirza & Osindero, 2014) | ✅ label is turned into a 28 × 28 map and joined to the image as an extra channel | real/fake | binary cross-entropy |
| **AC-GAN** (Odena, 2017) | ✅ D must **predict** the label with an extra output | real/fake **+** category | BCE **+** categorical cross-entropy |

**Expected:** with the Vanilla D nobody checks the category, so G can ignore the label → we cannot control what it
draws. cGAN and AC-GAN both force G to draw the requested category.

*(The last Dense layer outputs a raw score called a **logit**; `sigmoid(logit)` is the probability "real".
The loss functions use `from_logits=True`, which is numerically more stable.)*

In [ ]:
def conv_base(x):
    x = layers.Conv2D(64, 4, strides=2, padding="same")(x)    # 14 × 14
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Conv2D(128, 4, strides=2, padding="same")(x)   # 7 × 7
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)
    return layers.Flatten()(x)

def build_discriminator(kind):
    image = keras.Input(shape=(28, 28, 1), name="image")
    label = keras.Input(shape=(1,), dtype="int32", name="label")

    if kind == "vanilla":
        real_fake = layers.Dense(1)(conv_base(image))
        return keras.Model(image, real_fake, name="D_vanilla")

    if kind == "cgan":
        l = layers.Embedding(N_CLASSES, 50)(label)
        l = layers.Dense(28 * 28)(l)
        l = layers.Reshape((28, 28, 1))(l)
        x = layers.Concatenate()([image, l])                   # 28 × 28 × 2
        real_fake = layers.Dense(1)(conv_base(x))
        return keras.Model([image, label], real_fake, name="D_cgan")

    if kind == "acgan":
        features = conv_base(image)
        real_fake = layers.Dense(1, name="real_fake")(features)
        category = layers.Dense(N_CLASSES, name="category")(features)
        return keras.Model(image, [real_fake, category], name="D_acgan")

DISCRIMINATORS = ["vanilla", "cgan", "acgan"]
for kind in DISCRIMINATORS:
    d = build_discriminator(kind)
    print(f"{d.name:10s} parameters: {d.count_params():,}")

## 5. Loss functions and the training step (from scratch)
* **D loss** = BCE(D(real), 1) + BCE(D(fake), 0) → "say 1 for real, 0 for fake".
* **G loss** = BCE(D(fake), 1) → "make D say 1 for my fakes".
* **AC-GAN only:** both D and G also get a **category loss** (sparse categorical cross-entropy) — D must recognise the
  category of real and fake images, and G is rewarded when its fakes are recognised as the requested category.

`tf.GradientTape` records the computation so we can get gradients and update **only D** in step 1 and **only G** in step 2.
Both use the **Adam** optimizer with learning rate 0.0002 and beta_1 = 0.5 (the standard DCGAN settings).

In [ ]:
bce = keras.losses.BinaryCrossentropy(from_logits=True)
cce = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

LEARNING_RATE = 2e-4

def make_gan(kind):
    keras.utils.set_random_seed(SEED)
    G = build_generator()
    D = build_discriminator(kind)
    g_opt = keras.optimizers.Adam(LEARNING_RATE, beta_1=0.5)
    d_opt = keras.optimizers.Adam(LEARNING_RATE, beta_1=0.5)

    def run_D(images, labels):
        # Returns (real/fake logit, category logits or None)
        if kind == "vanilla":
            return D(images, training=True), None
        if kind == "cgan":
            return D([images, labels], training=True), None
        return D(images, training=True)                      # acgan → two outputs

    @tf.function
    def train_step(real_images, labels):
        batch = tf.shape(real_images)[0]
        noise = tf.random.normal([batch, LATENT_DIM])

        # ---- Step 1: train the discriminator
        with tf.GradientTape() as tape:
            fake_images = G([noise, labels], training=True)
            real_out, real_cls = run_D(real_images, labels)
            fake_out, fake_cls = run_D(fake_images, labels)
            d_loss = bce(tf.ones_like(real_out), real_out) + bce(tf.zeros_like(fake_out), fake_out)
            if kind == "acgan":
                d_loss += cce(labels, real_cls) + cce(labels, fake_cls)
        grads = tape.gradient(d_loss, D.trainable_variables)
        d_opt.apply_gradients(zip(grads, D.trainable_variables))

        # ---- Step 2: train the generator
        with tf.GradientTape() as tape:
            fake_images = G([noise, labels], training=True)
            fake_out, fake_cls = run_D(fake_images, labels)
            g_loss = bce(tf.ones_like(fake_out), fake_out)          # wants D to say "real"
            if kind == "acgan":
                g_loss += cce(labels, fake_cls)
        grads = tape.gradient(g_loss, G.trainable_variables)
        g_opt.apply_gradients(zip(grads, G.trainable_variables))
        return d_loss, g_loss

    return G, D, train_step

BATCH_SIZE = 128
dataset = (tf.data.Dataset.from_tensor_slices((x_train, y_train[:, None].astype("int32")))
           .shuffle(60000, seed=SEED).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE))

def train(train_step, epochs, on_epoch_end=None):
    history = []
    for epoch in range(1, epochs + 1):
        start = time.time()
        losses = [train_step(images, labels) for images, labels in dataset]
        d_loss, g_loss = np.mean([[float(d), float(g)] for d, g in losses], axis=0)
        history.append((d_loss, g_loss))
        print(f"epoch {epoch:2d} | D loss {d_loss:.3f} | G loss {g_loss:.3f} | {time.time() - start:.1f}s", flush=True)
        if on_epoch_end:
            on_epoch_end(epoch)
    return np.array(history)

# Every training run is kept: its models are saved to models/runs/<run name>/ and its settings and results are
# added as one row to training_log.csv — even runs that turn out worse, so there is proof of every experiment.
LOG_FILE = os.path.join(OUT_DIR, "training_log.csv")

def save_run(run_name, kind, epochs, G, D, history, seconds, class_acc, note=""):
    folder = os.path.join("models", "runs", run_name)
    os.makedirs(folder, exist_ok=True)
    G.save(os.path.join(folder, "generator.keras"))
    D.save(os.path.join(folder, "discriminator.keras"))
    row = pd.DataFrame([{
        "run": run_name, "discriminator": kind, "epochs": epochs, "batch size": BATCH_SIZE,
        "learning rate": LEARNING_RATE, "latent dim": LATENT_DIM,
        "final D loss": round(history[-1, 0], 4), "final G loss": round(history[-1, 1], 4),
        "class accuracy": round(class_acc, 4), "time (s)": round(seconds, 1),
        "finished at": time.strftime("%Y-%m-%d %H:%M"), "note": note,
    }])
    row.to_csv(LOG_FILE, mode="a", header=not os.path.exists(LOG_FILE), index=False)
    print(f"saved run '{run_name}' → {folder}")

# The same 6 noise vectors for every category: rows = categories, columns = noise vectors
fixed_noise = tf.tile(tf.random.normal([6, LATENT_DIM], seed=SEED), [N_CLASSES, 1])
fixed_labels = tf.constant(np.repeat(np.arange(N_CLASSES), 6)[:, None], tf.int32)

def show_samples(G, title, path=None, show=True):
    images = G([fixed_noise, fixed_labels], training=False).numpy()
    fig, axes = plt.subplots(N_CLASSES, 6, figsize=(6, 10))
    for i, ax in enumerate(axes.ravel()):
        ax.imshow(to_img(images[i]), cmap="gray"); ax.set_xticks([]); ax.set_yticks([])
        if i % 6 == 0:
            ax.set_ylabel(CLASS_NAMES[i // 6], rotation=0, ha="right", va="center", fontsize=8)
    plt.suptitle(title); plt.tight_layout()
    if path:
        fig.savefig(path, dpi=80)
    plt.show() if show else plt.close(fig)

## 6. A "judge" classifier to measure results
To compare the discriminators with a number, we train a normal CNN classifier on **real** images.
Then we generate images of each category and check **how often the judge recognises the category we asked for**
(*class accuracy*). Random guessing would give 10 %.

In [ ]:
judge = keras.Sequential([
    layers.Input((28, 28, 1)),
    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Flatten(), layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dense(N_CLASSES, activation="softmax"),
], name="judge")
judge.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
judge.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=2)
print(f"Judge accuracy on real test images: {judge.evaluate(x_test, y_test, verbose=0)[1]:.3f}")

def generate(G, label, n):
    return G([tf.random.normal([n, LATENT_DIM]), tf.fill([n, 1], label)], training=False).numpy()

def class_accuracy(G, n_per_class=300):
    images = np.concatenate([generate(G, c, n_per_class) for c in range(N_CLASSES)])
    labels = np.repeat(np.arange(N_CLASSES), n_per_class)
    return accuracy_score(labels, judge.predict(images, verbose=0).argmax(1))

## 7. Compare the three discriminators
Each GAN is trained for the **same 10 epochs** with the same generator and data. Only the discriminator changes.
All three trained models are **saved** (in `models/runs/`), not only the winner.

In [ ]:
CMP_EPOCHS = 10
comparison = {}
for kind in DISCRIMINATORS:
    print(f"\n===== {kind} =====")
    G_k, D_k, step = make_gan(kind)
    start = time.time()
    hist = train(step, CMP_EPOCHS)
    seconds, acc = time.time() - start, class_accuracy(G_k)
    comparison[kind] = {"class accuracy": acc, "final D loss": hist[-1, 0], "final G loss": hist[-1, 1],
                        "training time (s)": seconds}
    save_run(f"compare_{kind}_{CMP_EPOCHS}ep", kind, CMP_EPOCHS, G_k, D_k, hist, seconds, acc,
             note="discriminator comparison")
    show_samples(G_k, f"{kind} — after {CMP_EPOCHS} epochs", path=os.path.join(OUT_DIR, f"cmp_{kind}.png"))

comparison = pd.DataFrame(comparison).T
display(comparison.round(3))

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(comparison.index, comparison["class accuracy"], color=["tab:gray", "tab:blue", "tab:green"])
ax.axhline(0.1, ls="--", c="k", lw=1, label="random guess (10%)")
ax.set_ylabel("class accuracy"); ax.set_ylim(0, 1); ax.legend(); ax.set_title("Does G draw the requested category?")
plt.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "discriminator_comparison.png"), dpi=100); plt.show()

**What we learn**
* **Vanilla** — the images can look like clothes, but the rows do **not** match the requested category (class accuracy
  near 10 %). D never checks the label, so G learns to ignore it.
* **cGAN** — D sees the label, so G must draw the right category. Simple and effective.
* **AC-GAN** — D has to *classify* every image, and G is rewarded for drawing images that are easy to classify, so
  class accuracy is usually the highest. Its weakness: the images inside one category can look more alike (less variety).

We continue with the **cGAN** — it is the standard conditional GAN and gives a good balance of quality and variety.

## 8. Train the final cGAN
30 epochs. We save a picture after every epoch and turn them into an animation. This run is also saved and logged.

In [ ]:
EPOCHS = 30
G, D, train_step = make_gan("cgan")
start = time.time()
history = train(train_step, EPOCHS, on_epoch_end=lambda e: show_samples(
    G, f"epoch {e}", path=os.path.join(OUT_DIR, f"epoch_{e:02d}.png"), show=(e in (1, 10, EPOCHS))))
final_acc = class_accuracy(G)
save_run(f"final_cgan_{EPOCHS}ep", "cgan", EPOCHS, G, D, history, time.time() - start, final_acc, note="final model")

fig = plt.figure(figsize=(7, 3))
plt.plot(history[:, 0], label="D loss"); plt.plot(history[:, 1], label="G loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Training losses")
fig.savefig(os.path.join(OUT_DIR, "losses.png"), dpi=100); plt.show()

frames = [Image.open(os.path.join(OUT_DIR, f)) for f in sorted(os.listdir(OUT_DIR)) if f.startswith("epoch_")]
frames[0].save(os.path.join(OUT_DIR, "training.gif"), save_all=True, append_images=frames[1:], duration=300, loop=0)
print("Final class accuracy:", round(final_acc, 3))
display(pd.read_csv(LOG_FILE))                                # every run so far

**Reading the loss curve:** GAN losses do not go to zero like a normal classifier. D and G fight each other, so the
losses stay roughly level. If D loss dropped to ~0, D would be winning completely and G would stop learning.

## 9. Input vs output
### 9.1 What goes in and what comes out
The generator has **two inputs** — 100 random numbers (shown as a 10 × 10 grid of colours) and a category label —
and **one output**, a 28 × 28 image.

In [ ]:
examples = [CLASS_NAMES.index(c) for c in ["Sneaker", "Dress", "Bag"]]
noise = tf.random.normal([3, LATENT_DIM], seed=7)
outputs = G([noise, tf.constant(np.array(examples)[:, None])], training=False).numpy()

fig, axes = plt.subplots(3, 3, figsize=(10, 8), gridspec_kw={"width_ratios": [1.2, 1.4, 1]})
for r, c in enumerate(examples):
    axes[r, 0].imshow(noise[r].numpy().reshape(10, 10), cmap="coolwarm", vmin=-2.5, vmax=2.5)
    axes[r, 0].set_title("INPUT 1: noise z (100 numbers)", fontsize=8); axes[r, 0].axis("off")
    onehot = np.eye(N_CLASSES)[c]
    axes[r, 1].barh(CLASS_NAMES, onehot, color=["tab:purple" if v else "lightgray" for v in onehot])
    axes[r, 1].invert_yaxis(); axes[r, 1].set_xticks([]); axes[r, 1].tick_params(labelsize=7)
    axes[r, 1].set_title("INPUT 2: category label", fontsize=8)
    axes[r, 2].imshow(to_img(outputs[r]), cmap="gray"); axes[r, 2].axis("off")
    axes[r, 2].set_title(f"OUTPUT: generated {CLASS_NAMES[c]}", fontsize=8)
plt.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "input_output.png"), dpi=100); plt.show()

In [ ]:
# Which input controls what?
fig, axes = plt.subplots(2, 10, figsize=(13, 3.2))
same_noise = tf.tile(tf.random.normal([1, LATENT_DIM], seed=11), [10, 1])
row1 = G([same_noise, tf.constant(np.arange(10)[:, None])], training=False).numpy()
row2 = generate(G, CLASS_NAMES.index("Sneaker"), 10)
for i in range(10):
    axes[0, i].imshow(to_img(row1[i]), cmap="gray"); axes[0, i].axis("off"); axes[0, i].set_title(CLASS_NAMES[i], fontsize=7)
    axes[1, i].imshow(to_img(row2[i]), cmap="gray"); axes[1, i].axis("off")
axes[0, 0].text(-40, 14, "same noise,\nchange label", fontsize=8, va="center")
axes[1, 0].text(-40, 14, "same label,\nchange noise", fontsize=8, va="center")
plt.suptitle("The LABEL decides WHAT is drawn — the NOISE decides the STYLE")
plt.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "label_vs_noise.png"), dpi=100); plt.show()

### 9.2 Real images vs generated images
**Average image per category** — real average, generated average, and the **absolute difference** between them
(bright = big difference). Small differences mean G learned the typical shape of each category.

In [ ]:
gen_images = np.concatenate([generate(G, c, 500) for c in range(N_CLASSES)])
gen_labels = np.repeat(np.arange(N_CLASSES), 500)

fig, axes = plt.subplots(3, 10, figsize=(13, 4.5))
for c in range(N_CLASSES):
    real_mean = x_train[y_train == c].mean(axis=0)
    gen_mean = gen_images[gen_labels == c].mean(axis=0)
    axes[0, c].imshow(to_img(real_mean), cmap="gray", vmin=0, vmax=1)
    axes[1, c].imshow(to_img(gen_mean), cmap="gray", vmin=0, vmax=1)
    axes[2, c].imshow(np.abs(np.squeeze(real_mean - gen_mean)) / 2, cmap="hot", vmin=0, vmax=0.3)
    axes[0, c].set_title(CLASS_NAMES[c], fontsize=7)
    for r in range(3):
        axes[r, c].axis("off")
for r, name in enumerate(["real average", "generated average", "difference"]):
    axes[r, 0].text(-3, 14, name, fontsize=8, ha="right", va="center")
plt.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "mean_difference.png"), dpi=100); plt.show()

### 9.3 Is the GAN copying training images?
For each generated image we find the **most similar real training image** (smallest pixel distance) and show the
difference. If the GAN were just copying, the difference would be almost black.

In [ ]:
train_flat = x_train.reshape(len(x_train), -1)

def nearest_real(image):
    distances = np.sqrt(((train_flat - image.reshape(1, -1)) ** 2).sum(axis=1))
    return x_train[distances.argmin()], distances.min()

fig, axes = plt.subplots(3, 10, figsize=(13, 4.5))
for c in range(N_CLASSES):
    gen = gen_images[gen_labels == c][0]
    near, dist = nearest_real(gen)
    axes[0, c].imshow(to_img(gen), cmap="gray")
    axes[1, c].imshow(to_img(near), cmap="gray")
    axes[2, c].imshow(np.abs(np.squeeze(gen - near)) / 2, cmap="hot", vmin=0, vmax=0.5)
    axes[0, c].set_title(CLASS_NAMES[c], fontsize=7); axes[2, c].set_title(f"distance {dist:.1f}", fontsize=7)
    for r in range(3):
        axes[r, c].axis("off")
for r, name in enumerate(["generated", "closest real", "difference"]):
    axes[r, 0].text(-3, 14, name, fontsize=8, ha="right", va="center")
plt.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "nearest_difference.png"), dpi=100); plt.show()

### 9.4 What does the discriminator think?
D outputs the probability that an image is real. We give it real test images, generated images and random noise.

In [ ]:
def d_probability(images, labels):
    return tf.sigmoid(D.predict([images, labels[:, None]], verbose=0)).numpy().ravel()

random_noise = np.random.uniform(-1, 1, (1000, 28, 28, 1)).astype("float32")
p_real = d_probability(x_test[:1000], y_test[:1000])
p_fake = d_probability(gen_images[::5], gen_labels[::5])
p_noise = d_probability(random_noise, np.random.randint(0, 10, 1000))

fig = plt.figure(figsize=(7, 3))
plt.hist(p_real, bins=30, alpha=0.6, label=f"real images (avg {p_real.mean():.2f})")
plt.hist(p_fake, bins=30, alpha=0.6, label=f"generated images (avg {p_fake.mean():.2f})")
plt.hist(p_noise, bins=30, alpha=0.6, label=f"random noise (avg {p_noise.mean():.2f})")
plt.xlabel("D's probability that the image is real"); plt.legend(fontsize=8)
fig.savefig(os.path.join(OUT_DIR, "discriminator_scores.png"), dpi=100, bbox_inches="tight"); plt.show()

If G were perfect, D could not tell real from generated, and both would sit around 0.5. Random noise should get a
much lower probability.

## 10. How well does the judge recognise each generated category?

In [ ]:
pred = judge.predict(gen_images, verbose=0).argmax(1)
real_pred = judge.predict(x_test, verbose=0).argmax(1)
fidelity = pd.DataFrame({
    "judge accuracy on real images": [accuracy_score(y_test[y_test == c], real_pred[y_test == c]) for c in range(N_CLASSES)],
    "judge accuracy on generated images": [accuracy_score(gen_labels[gen_labels == c], pred[gen_labels == c]) for c in range(N_CLASSES)],
}, index=CLASS_NAMES)
fidelity.loc["ALL"] = [accuracy_score(y_test, real_pred), accuracy_score(gen_labels, pred)]
display(fidelity.round(3))

cm = confusion_matrix(gen_labels, pred, normalize="true")
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(10), CLASS_NAMES, rotation=60, ha="right", fontsize=8); ax.set_yticks(range(10), CLASS_NAMES, fontsize=8)
ax.set_xlabel("judge's prediction"); ax.set_ylabel("category we asked for")
for i in range(10):
    for j in range(10):
        if cm[i, j] >= 0.05:
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if cm[i, j] > 0.5 else "black")
ax.set_title("Confusion matrix for generated images")
plt.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=100); plt.show()

Categories that look alike (T-shirt, shirt, pullover, coat) get confused most — the judge confuses them on **real**
images too.

## 11. Save the models

In [ ]:
os.makedirs("models", exist_ok=True)
G.save("models/fashion_generator.keras")
D.save("models/fashion_discriminator.keras")
judge.save("models/fashion_judge.keras")
# 300 real images per category, used by the app to show "closest real image"
keep = np.concatenate([np.where(y_train == c)[0][:300] for c in range(N_CLASSES)])
np.savez_compressed("models/real_samples.npz", x=((x_train[keep, :, :, 0] + 1) * 127.5).round().astype(np.uint8),
                    y=y_train[keep])
comparison.to_csv(os.path.join(OUT_DIR, "discriminator_comparison.csv"))
fidelity.to_csv(os.path.join(OUT_DIR, "fidelity.csv"))
print("Saved.")

## Summary
* A **GAN** = generator (makes fakes from noise) vs discriminator (spots fakes), trained in turns with binary cross-entropy.
* A **conditional GAN** also gets the **category label**, so we choose what is generated.
* We built and compared **three discriminators**: Vanilla (ignores the label → no control), **cGAN** (label as an extra
  input channel) and **AC-GAN** (D also predicts the label).
* **Input vs output:** the label decides *what* is drawn, the noise decides the *style*; generated images match the
  average shape of real ones but are **not copies** of training images.

**Limitations:** small 28 × 28 grayscale images; similar categories (shirt vs T-shirt) are sometimes mixed up.

### Download the results (Colab)

In [ ]:
import shutil
os.makedirs("gan_results", exist_ok=True)
shutil.copytree(OUT_DIR, "gan_results/gan_outputs", dirs_exist_ok=True)
shutil.copytree("models", "gan_results/models", dirs_exist_ok=True)
shutil.make_archive("fashion_gan_results", "zip", "gan_results")
try:
    from google.colab import files
    files.download("fashion_gan_results.zip")
except ImportError:
    print("Saved fashion_gan_results.zip")